# Brain MRI Tumor Segmentation: BaselineCNN vs U-Net vs Pretrained U-Net

**Project:** Clinical Decision Support System (CDSS) for MRI brain tumor segmentation
**Dataset:** Kaggle LGG MRI Segmentation (110 TCGA patients, 3,929 slices)

Same pipeline as the original notebook (`archive/code_file_original.ipynb`); the code now lives in `src/`.

1. Setup and config
2. EDA dashboard
3. Data split, preprocessing and augmentation
4. Train the 5 main experiments
5. Results, plots and predictions
6. Standalone `unet_pretrained_aug`
7. Augmentation diagnostics

## 0. Setup

In [ ]:
# !pip install -r ../requirements.txt --quiet      # first time only
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)

import pandas as pd

from src.config import Config
from src.utils import setup_torch, seed_everything, get_device, ensure_dirs
from src.eda import generate_clinical_dashboard
from src.data import build_data
from src.models import BaselineCNN, UNet, build_pretrained_unet
from src.train import run_experiment, save_results, load_results
from src.metrics import per_image_dice, bootstrap_ci
from src.plots import plot_comparison, plot_convergence, visualize_predictions

cfg = Config()
# cfg.data_dir = "/content/kaggle_3m"   # dataset path (default: data/kaggle_3m)
# cfg.limit = 200; cfg.epochs = 2      # quick smoke test

setup_torch()
seed_everything(cfg.seed)
device = get_device()
ensure_dirs(cfg.out_dir, cfg.ckpt_dir)
print(f"Using device: {device}")

## 1. EDA dashboard

In [ ]:
generate_clinical_dashboard(cfg.data_dir, os.path.join(cfg.out_dir, "clinical_eda_results_final.png"), show=True)

## 2. Data split + transforms + loaders
70 / 15 / 15 split stratified by tumor / no-tumor. Resize 256, ImageNet normalization.
Augmentation: CLAHE, ElasticTransform, ShiftScaleRotate (masks with nearest interpolation), flip, brightness/contrast.

In [ ]:
ds, loaders = build_data(cfg, device)

## 3. Run the 5 main experiments

In [ ]:
experiments = [
    ("baseline_plain", BaselineCNN(), loaders["train_plain"]),
    ("baseline_aug", BaselineCNN(), loaders["train_aug"]),
    ("unet_plain", UNet(), loaders["train_plain"]),
    ("unet_aug", UNet(), loaders["train_aug"]),
]
if not cfg.skip_pretrained:
    experiments.append(("unet_pretrained_plain", build_pretrained_unet(), loaders["train_plain"]))

all_histories, final_results, trained_models, best_thresholds = {}, {}, {}, {}
for name, model, loader in experiments:
    trained_model, history, row = run_experiment(name, model, loader, cfg, ds, loaders, device)
    all_histories[name] = history
    trained_models[name] = trained_model
    best_thresholds[name] = row["best_threshold"]
    final_results[name] = row

## 4. Save results + plots

In [ ]:
results_df, _ = save_results(cfg.out_dir, final_results, all_histories)
print("FINAL RESULTS SUMMARY (test set, tuned threshold + blob filter)")
results_df.astype(float).round(4)

In [ ]:
plot_comparison(all_histories, os.path.join(cfg.out_dir, "training_comparison.png"), show=True)

best_model_name = max(final_results, key=lambda k: final_results[k]["test_dice"])
print(f"Best model on test set: {best_model_name} (Dice={final_results[best_model_name]['test_dice']:.4f})")
visualize_predictions(trained_models[best_model_name], ds["test"], device,
                      os.path.join(cfg.out_dir, f"predictions_{best_model_name}.png"),
                      thresh=best_thresholds[best_model_name], use_blob_filter=True,
                      min_blob_size=cfg.min_blob_size, show=True)

## 5. Run just unet_pretrained_aug (standalone)

In [ ]:
name = "unet_pretrained_aug"
trained_model, history, row = run_experiment(name, build_pretrained_unet(), loaders["train_aug"],
                                             cfg, ds, loaders, device)
all_histories[name] = history
trained_models[name] = trained_model
best_thresholds[name] = row["best_threshold"]
final_results[name] = row

results_df, _ = save_results(cfg.out_dir, {name: row}, {name: history})
print("Updated results:")
results_df.astype(float).round(4)

## 6. Diagnostic 1: did augmented training actually converge?
If the aug curve is still rising (val_dice) when training stops, it was cut off early rather than genuinely worse.

In [ ]:
plot_convergence(all_histories, "unet_plain", "unet_aug",
                 os.path.join(cfg.out_dir, "diag1_convergence_unet.png"), show=True)

## 7. Diagnostic 2: re-run aug with more patience + softer augmentation
epochs 25 -> 40, patience 6 -> 12, softer ElasticTransform / ShiftScaleRotate.

In [ ]:
name = "unet_aug_soft_longtrain"
trained_model, history, row = run_experiment(name, UNet(), loaders["train_aug_soft"], cfg, ds, loaders,
                                             device, epochs=40, patience=12)
print("Compare against:")
print(f"  unet_plain   test_dice = {final_results['unet_plain']['test_dice']:.4f}")
print(f"  unet_aug     test_dice = {final_results['unet_aug']['test_dice']:.4f}")
print(f"  {name} test_dice = {row['test_dice']:.4f}")

all_histories[name] = history
final_results[name] = row
trained_models[name] = trained_model
save_results(cfg.out_dir, {name: row}, {name: history});

## 8. Diagnostic 3: is the gap statistically meaningful?
Bootstrap the per-image Dice on the test set. If the plain vs aug CIs overlap a lot, the gap may just be noise.

In [ ]:
plain_dices = per_image_dice(trained_models["unet_plain"], ds["test"], device)
aug_dices = per_image_dice(trained_models["unet_aug"], ds["test"], device)

mean_p, lo_p, hi_p = bootstrap_ci(plain_dices)
mean_a, lo_a, hi_a = bootstrap_ci(aug_dices)

print(f"unet_plain: mean={mean_p:.4f}  95% CI=[{lo_p:.4f}, {hi_p:.4f}]")
print(f"unet_aug:   mean={mean_a:.4f}  95% CI=[{lo_a:.4f}, {hi_a:.4f}]")
if hi_a > lo_p:
    print("-> CIs overlap: the gap may not be statistically robust from one run alone.")
else:
    print("-> CIs do NOT overlap: this looks like a real, reproducible effect.")